Script for batch inference with OpenAI API, GPT-5 mini model

In [ ]:
from dotenv import load_dotenv
import os
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

load_dotenv(dotenv_path=ROOT / "data" / ".env")

openai_api_key = os.getenv("OPENAI_API_KEY")

Generate data with OpenAI

In [4]:
from openai import OpenAI

counselor = OpenAI(api_key=openai_api_key)

In [ ]:
import json
import os
import pandas as pd
from datasets import load_dataset

questions = load_dataset("rileyhitthefan/age-based-health-qa")["train"]
questions = pd.DataFrame(questions)

os.makedirs(ROOT / "data" / "batch", exist_ok=True)
with open(ROOT / "data" / "batch" / "gpt.jsonl", "w") as out:
    for i, row in questions.iterrows():
        
        entry = {
            "custom_id": f"req-{i}",
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": "gpt-5-mini",
                "messages": [
                    {"role": "system", "content": "You are a helpful assistant. Answer the user's questions in 1 clear and concise paragraph."},
                    {"role": "user", "content": row['prompt']}
                ],
            "max_completion_tokens": 1200
            }
        }
        out.write(json.dumps(entry) + "\n")

Poll batch job until completion

In [ ]:
with open(ROOT / "data" / "batch" / "gpt.jsonl", "rb") as batch_file:
    batch_input_file = counselor.files.create(
        file=batch_file,
        purpose="batch"
    )

batch = counselor.batches.create(
    input_file_id=batch_input_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={
        "description": "batch job"
    }
)

batch

Batch(id='batch_68ec830a308c81909cdd7b1a40c222db', completion_window='24h', created_at=1760330506, endpoint='/v1/chat/completions', input_file_id='file-1AMKENA1orkLBazNVzymrZ', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1760416906, failed_at=None, finalizing_at=None, in_progress_at=None, metadata={'description': 'batch job'}, model=None, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0), usage=BatchUsage(input_tokens=0, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=0, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=0))

In [ ]:
import os
import time

poll_interval = 300

while True:
    output = counselor.batches.retrieve(batch.id)
    print(f"Current status: {output.status}")

    if output.status in ["completed", "failed", "cancelled"]:
        break

    time.sleep(poll_interval)

messages = []
result_text = ""

if output.status == "completed":
    result_file = counselor.files.content(output.output_file_id)
    result_text = result_file.read().decode("utf-8", errors="ignore").strip()

    if output.error_file_id:
        os.makedirs(ROOT / "data" / "responses", exist_ok=True)
        error_file = counselor.files.content(output.error_file_id)
        with open(ROOT / "data" / "responses" / "gpt_errors.jsonl", "wb") as f:
            f.write(error_file.read())

    for line in result_text.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            item = json.loads(line)
            choice = item["response"]["body"]["choices"][0]
            message = choice.get("message", {})
            if message.get("role") == "assistant" and message.get("content"):
                messages.append(message["content"])
            else:
                messages.append("No response")
        except Exception:
            messages.append("No response")
            continue

os.makedirs(ROOT / "data" / "responses", exist_ok=True)
with open(ROOT / "data" / "responses" / "gpt.txt", "w", encoding="utf-8") as f:
    for i, message in enumerate(messages):
        f.write(str(i + 1) + ". " + message + "\n")

Current status: validating
Current status: in_progress
Current status: in_progress
Current status: in_progress
Current status: finalizing
Current status: completed


In [ ]:
records = []

if not result_text:
    raise RuntimeError("No batch output text; run the poll cell after status is completed.")

for line in result_text.splitlines():
    line = line.strip()
    if not line:
        continue
    try:
        obj = json.loads(line)
        records.append(obj)
    except Exception:
        continue

with open(ROOT / "data" / "responses" / "gpt.jsonl", "w", encoding="utf-8") as f:
    for record in records:
        json.dump(record, f, ensure_ascii=False)
        f.write("\n")

print(f"Saved {len(records)} responses.")

Saved 6000 responses.


In [9]:
output

Batch(id='batch_68ec830a308c81909cdd7b1a40c222db', completion_window='24h', created_at=1760330506, endpoint='/v1/chat/completions', input_file_id='file-1AMKENA1orkLBazNVzymrZ', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1760331916, error_file_id=None, errors=None, expired_at=None, expires_at=1760416906, failed_at=None, finalizing_at=1760331667, in_progress_at=1760330570, metadata={'description': 'batch job'}, model='gpt-5-mini-2025-08-07', output_file_id='file-Lu2iaTttvsqnvHjBAu8SgN', request_counts=BatchRequestCounts(completed=6000, failed=0, total=6000), usage=BatchUsage(input_tokens=386734, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=3337798, output_tokens_details=OutputTokensDetails(reasoning_tokens=2091936), total_tokens=3724532))

In [10]:
questions["response_gpt"] = messages
len(messages)

6000

In [ ]:
questions.to_csv(ROOT / "data" / "responses" / "response_gpt.csv", index=False)